In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


# -----------------------------
# 1. Generate better dataset
# -----------------------------
np.random.seed(0)

# Continuous hours make the tree boundaries easier to see
hours = np.random.uniform(1, 10, 80)

# Real rule: students usually pass if hours >= 6
pass_exam = (hours >= 6).astype(int)

# Add noise: flip some labels
#noise_index = np.random.choice(len(pass_exam), 12, replace=False)

#Add more dramatic noise
noise_index = np.random.choice(len(pass_exam), 18, replace=False)

pass_exam[noise_index] = 1 - pass_exam[noise_index]

data = pd.DataFrame({
    "hours": hours,
    "pass": pass_exam
})

X = data[["hours"]]
y = data["pass"]


# -----------------------------
# 2. Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)


# -----------------------------
# 3. Train models
# -----------------------------
underfit_model = DecisionTreeClassifier(max_depth=1, random_state=42)
underfit_model.fit(X_train, y_train)

good_model = DecisionTreeClassifier(max_depth=3, random_state=42)
good_model.fit(X_train, y_train)

overfit_model = DecisionTreeClassifier(random_state=42)
overfit_model.fit(X_train, y_train)


# -----------------------------
# 4. Print scores
# -----------------------------
print("Underfit Train:", round(underfit_model.score(X_train, y_train), 3))
print("Underfit Test :", round(underfit_model.score(X_test, y_test), 3))

print("Good Fit Train:", round(good_model.score(X_train, y_train), 3))
print("Good Fit Test :", round(good_model.score(X_test, y_test), 3))

print("Overfit Train:", round(overfit_model.score(X_train, y_train), 3))
print("Overfit Test :", round(overfit_model.score(X_test, y_test), 3))


# -----------------------------
# 5. Create plotting range
# -----------------------------
x_range = np.linspace(X["hours"].min(), X["hours"].max(), 500).reshape(-1, 1)

underfit_pred = underfit_model.predict(x_range)
good_pred = good_model.predict(x_range)
overfit_pred = overfit_model.predict(x_range)

# -----------------------------
# 6. Plot
# -----------------------------


# scatter actual data
# jitter added vertically so overlapping points are easier to see
y_jitter = y + np.random.uniform(-0.05, 0.05, size=len(y))


fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

models = [
    ("Underfit (max_depth=1)", underfit_model, underfit_pred),
    ("Good Fit (max_depth=3)", good_model, good_pred),
    ("Overfit (no limit)", overfit_model, overfit_pred)
]

for ax, (title, model, pred) in zip(axes, models):
    ax.scatter(X["hours"], y_jitter, alpha=0.7)
    ax.step(x_range.ravel(), pred, where="mid", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Hours Studied")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Pass / Fail")
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["Fail", "Pass"])

plt.suptitle("Decision Tree Classification Comparison", fontsize=14)
plt.tight_layout()
plt.show()